# Modular Breakdown of Baseline Pipeline

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score
import numpy as np

# # Function for loading and splitting data
# def load_and_split_data(csv_path, test_size=0.3, random_state=42):
#     df = pd.read_csv(csv_path)
#     print("\n--- Class Distribution ---")
#     print(df['label'].value_counts())

#     train_df, temp_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df['label'])
#     val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=random_state, stratify=temp_df['label'])

#     print(f"\n--- Dataset Split ---")
#     print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
#     return train_df, val_df, test_df

# function for loading already split data
def load_presplit_data(train_csv_path, val_csv_path, test_csv_path):
    train_df = pd.read_csv(train_csv_path)
    val_df = pd.read_csv(val_csv_path)
    test_df = pd.read_csv(test_csv_path)

    print("\n--- Loaded Pre-split Data ---")
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    return train_df, val_df, test_df

# Function for TF-IDF feature extraction
def extract_tfidf_features(train_df, val_df, test_df, max_features=50000, ngram_range=(1, 2)):
    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=True,
        strip_accents='unicode',
        analyzer='word'
    )

    X_train_tfidf = tfidf.fit_transform(train_df['text'])
    X_val_tfidf = tfidf.transform(val_df['text'])
    X_test_tfidf = tfidf.transform(test_df['text'])

    y_train = train_df['Sentiment'].values
    y_val = val_df['Sentiment'].values
    y_test = test_df['Sentiment'].values

    print("\n--- TF-IDF Feature Extraction Complete ---")
    return X_train_tfidf, X_val_tfidf, X_test_tfidf, y_train, y_val, y_test, tfidf

# Function to evaluate a model
def evaluate_model(model, X_test, y_test, model_name, run):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"\n{'='*50}")
    print(f"  {model_name} | Run {run}")
    print(f"{'='*50}")
    print(report)
    return f1

# Function for training and evaluating classical models
def train_and_evaluate_classical_models(X_train_tfidf, y_train, X_test_tfidf, y_test, seeds=[42, 7]):
    results = {}
    for run, seed in enumerate(seeds, start=1):
        print(f"\n--- Training Models for Run {run} (seed: {seed}) ---")

        # Logistic Regression
        lr = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
        lr.fit(X_train_tfidf, y_train)
        results[f'LR_run{run}'] = evaluate_model(lr, X_test_tfidf, y_test, "Logistic Regression", run)

        # Support Vector Machine
        svm = LinearSVC(max_iter=2000, C=1.0, random_state=seed)
        svm.fit(X_train_tfidf, y_train)
        results[f'SVM_run{run}'] = evaluate_model(svm, X_test_tfidf, y_test, "SVM", run)

        # Multinomial Naive Bayes
        mnb = MultinomialNB(alpha=0.1)
        mnb.fit(X_train_tfidf, y_train)
        results[f'MNB_run{run}'] = evaluate_model(mnb, X_test_tfidf, y_test, "Naive Bayes", run)

    print("\n--- All Classical Models Training and Evaluation Complete ---")
    return results

The previous code has been refactored into the following functions:

*   `load_and_split_data(csv_path, test_size, random_state)`: Loads a CSV, splits it into train/validation/test sets.
*   `extract_tfidf_features(train_df, val_df, test_df, max_features, ngram_range)`: Applies TF-IDF vectorization to the text data.
*   `train_and_evaluate_classical_models(X_train_tfidf, y_train, X_test_tfidf, y_test, seeds)`: Trains Logistic Regression, SVM, and Multinomial Naive Bayes models and evaluates them.

Now, let's execute these functions in sequence to demonstrate their usage.

In [7]:
# Example usage of the modularized functions

# Step 1: Install dependencies (if not already installed)
!pip install transformers datasets scikit-learn torch pandas numpy seaborn matplotlib evaluate accelerate

# Step 2: Load and prepare your dataset
# train_df, val_df, test_df = load_and_split_data("your_dataset.csv") # Original call commented out

# New call for loading pre-split data
train_df, val_df, test_df = load_presplit_data(
    "/content/train.csv",
    "/content/validation.csv",
    "/content/test.csv"
)

# Step 3: TF-IDF Feature Extraction
X_train_tfidf, X_val_tfidf, X_test_tfidf, y_train, y_val, y_test, tfidf_vectorizer = extract_tfidf_features(train_df, val_df, test_df)

# Step 4: Train and Evaluate Classical Baselines
classical_results = train_and_evaluate_classical_models(X_train_tfidf, y_train, X_test_tfidf, y_test)

print("\n--- Final Classical Model Results ---")
print(classical_results)


--- Loaded Pre-split Data ---
Train: 3747, Val: 313, Test: 2183

--- TF-IDF Feature Extraction Complete ---

--- Training Models for Run 1 (seed: 42) ---

  Logistic Regression | Run 1
              precision    recall  f1-score   support

         0.0     0.8138    0.8568    0.8347      1117
         1.0     0.8411    0.7946    0.8172      1066

    accuracy                         0.8264      2183
   macro avg     0.8274    0.8257    0.8259      2183
weighted avg     0.8271    0.8264    0.8261      2183


  SVM | Run 1
              precision    recall  f1-score   support

         0.0     0.8354    0.8630    0.8490      1117
         1.0     0.8513    0.8218    0.8363      1066

    accuracy                         0.8429      2183
   macro avg     0.8433    0.8424    0.8426      2183
weighted avg     0.8431    0.8429    0.8428      2183


  Naive Bayes | Run 1
              precision    recall  f1-score   support

         0.0     0.8205    0.7941    0.8071      1117
         1.0 